# Solar Radiation Forecasting

Predict surface solar irradiance using Earth2Studio's SolarRadiationAFNO diagnostic model. Useful for solar energy planning and grid operations.

**Extra install:** `uv add earth2studio --extra solarradiation-afno`

In [ ]:
import os
from datetime import datetime, timedelta

import torch
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature

import earth2studio.run as run
from earth2studio.models.px import FCN
from earth2studio.models.dx import SolarRadiationAFNO6H
from earth2studio.data import GFS
from earth2studio.io import ZarrBackend

In [ ]:
CONFIG = {
    "forecast_date": "2026-06-21",  # summer solstice — peak irradiance variety
    "nsteps": 16,                   # 4 days at 6h intervals — full diurnal cycles
    "output_root": "outputs/solar",
    "device": "cuda" if torch.cuda.is_available() else "cpu",
}

os.makedirs(CONFIG["output_root"], exist_ok=True)
print(f"Device: {CONFIG['device']}")

## Run Solar Radiation Forecast

In [ ]:
fcn_model = FCN.load_model(FCN.load_default_package())
solar_model = SolarRadiationAFNO6H.load_model(SolarRadiationAFNO6H.load_default_package())
gfs_data = GFS()

io = ZarrBackend(
    f"{CONFIG['output_root']}/fcn_solar.zarr",
    backend_kwargs={"overwrite": True},
)
io = run.diagnostic(
    [CONFIG["forecast_date"]], CONFIG["nsteps"],
    fcn_model, solar_model, gfs_data, io,
)
ds = xr.open_zarr(f"{CONFIG['output_root']}/fcn_solar.zarr")

print("Variables:", list(ds.data_vars))
for var in ds.data_vars:
    print(f"  {var}: {ds[var].shape}")

In [ ]:
# Identify the solar radiation variable
solar_vars = [v for v in ds.data_vars if "sr" in v.lower() or "rad" in v.lower()
              or "ssrd" in v.lower() or "sw" in v.lower()]
if solar_vars:
    SOLAR_VAR = solar_vars[0]
else:
    # Fallback: pick the first non-standard variable
    standard = {"t2m", "u10m", "v10m", "msl", "z500", "tp", "sp", "tcwv"}
    SOLAR_VAR = [v for v in ds.data_vars if v not in standard][0]

lats = ds["lat"].values
lons = ds["lon"].values
print(f"Solar variable: {SOLAR_VAR}")

## Global Irradiance Maps

Four time steps showing the diurnal cycle — the day/night terminator is clearly visible.

In [ ]:
steps = [0, 2, 4, 6]  # 0h, +12h, +24h, +36h — captures day/night transitions
fig, axes = plt.subplots(2, 2, figsize=(18, 12), subplot_kw={"projection": ccrs.Robinson()})

for ax, step in zip(axes.flat, steps):
    lead_hours = step * 6
    solar = ds[SOLAR_VAR].isel(time=0, lead_time=step).values

    cf = ax.pcolormesh(lons, lats, solar, transform=ccrs.PlateCarree(),
                       cmap="YlOrRd", vmin=0, shading="auto")
    ax.add_feature(cfeature.COASTLINE, linewidth=0.6)
    ax.gridlines(linewidth=0.3, alpha=0.3)
    ax.set_title(f"+{lead_hours}h", fontsize=12)

fig.suptitle(f"Surface Solar Irradiance  |  Init: {CONFIG['forecast_date']}", fontsize=14, y=0.98)
cbar = fig.colorbar(cf, ax=axes.ravel().tolist(), orientation="horizontal",
                    pad=0.04, shrink=0.5, label=f"{SOLAR_VAR} (W/m²)")
plt.savefig(f"{CONFIG['output_root']}/global_irradiance.png", dpi=150, bbox_inches="tight")
plt.show()

## Diurnal Cycle at Key Locations

In [ ]:
# Define locations of interest
locations = {
    "Phoenix, AZ": (33.45, -112.07),
    "Seattle, WA": (47.61, -122.33),
    "London, UK": (51.51, -0.13),
    "Sahara": (25.0, 10.0),
}

# Extract timeseries at each location
hours = np.arange(CONFIG["nsteps"] + 1) * 6

fig, ax = plt.subplots(figsize=(12, 5))
colors = plt.cm.tab10(np.linspace(0, 1, len(locations)))

for (name, (lat, lon)), color in zip(locations.items(), colors):
    lon_360 = lon % 360
    lat_idx = np.argmin(np.abs(lats - lat))
    lon_idx = np.argmin(np.abs(lons - lon_360))

    solar_ts = ds[SOLAR_VAR].isel(time=0, lat=lat_idx, lon=lon_idx).values
    ax.plot(hours[:len(solar_ts)], solar_ts, marker="o", markersize=4,
            label=name, color=color, linewidth=2)

ax.set_xlabel("Lead Time (hours)")
ax.set_ylabel(f"{SOLAR_VAR} (W/m²)")
ax.set_title(f"Solar Irradiance Diurnal Cycle  |  Init: {CONFIG['forecast_date']}")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(f"{CONFIG['output_root']}/diurnal_cycle.png", dpi=150, bbox_inches="tight")
plt.show()

## Daily Solar Energy Potential

Integrate irradiance over 24 hours to estimate daily energy yield per grid cell (kWh/m²).

In [ ]:
# Sum irradiance over first 4 steps (24 hours at 6h intervals)
# Energy = irradiance * time_interval (6h = 21600s), converted to kWh
solar_day = np.zeros_like(ds[SOLAR_VAR].isel(time=0, lead_time=0).values)
for s in range(min(4, CONFIG["nsteps"] + 1)):
    solar_day += ds[SOLAR_VAR].isel(time=0, lead_time=s).values

# Convert: W/m2 * 6h * 4_steps = W*h/m2, then /1000 = kWh/m2
daily_kwh = solar_day * 6 / 1000  # approximate daily energy

fig, ax = plt.subplots(figsize=(14, 7), subplot_kw={"projection": ccrs.Robinson()})
cf = ax.pcolormesh(lons, lats, daily_kwh, transform=ccrs.PlateCarree(),
                   cmap="YlOrRd", vmin=0, shading="auto")
cbar = plt.colorbar(cf, ax=ax, orientation="horizontal", pad=0.04, shrink=0.6)
cbar.set_label("Daily Solar Energy (kWh/m²)")
ax.add_feature(cfeature.COASTLINE, linewidth=0.7)
ax.add_feature(cfeature.BORDERS, linewidth=0.4, alpha=0.5)
ax.gridlines(linewidth=0.3, alpha=0.4)
ax.set_title(f"Estimated Daily Solar Energy Potential  |  {CONFIG['forecast_date']}")
plt.tight_layout()
plt.savefig(f"{CONFIG['output_root']}/daily_energy.png", dpi=150, bbox_inches="tight")
plt.show()

## Regional Focus: US Southwest

In [ ]:
fig, ax = plt.subplots(figsize=(12, 8), subplot_kw={"projection": ccrs.PlateCarree()})
extent = [-125, -100, 25, 45]

step = 2  # midday step
solar = ds[SOLAR_VAR].isel(time=0, lead_time=step).values

cf = ax.pcolormesh(lons, lats, solar, transform=ccrs.PlateCarree(),
                   cmap="YlOrRd", vmin=0, shading="auto")
ax.set_extent(extent, crs=ccrs.PlateCarree())
ax.add_feature(cfeature.COASTLINE, linewidth=0.8)
ax.add_feature(cfeature.STATES, linewidth=0.5, alpha=0.5)
ax.add_feature(cfeature.BORDERS, linewidth=0.6)
gl = ax.gridlines(draw_labels=True, linewidth=0.3, alpha=0.4)
gl.top_labels = gl.right_labels = False

cbar = plt.colorbar(cf, ax=ax, orientation="vertical", pad=0.02, shrink=0.8)
cbar.set_label(f"{SOLAR_VAR} (W/m²)")
ax.set_title(f"Solar Irradiance — US Southwest  |  +{step*6}h  |  Init: {CONFIG['forecast_date']}")
plt.tight_layout()
plt.savefig(f"{CONFIG['output_root']}/southwest_solar.png", dpi=150, bbox_inches="tight")
plt.show()